In [ ]:
!pip uninstall torch -y
!pip install torch==2.8.0
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster
!pip install torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.0+cu121.html

In [ ]:
from google.colab import drive
# First, download the files from the GitHub repository and upload the folder to your Google Drive.
# Then, enter the path to that folder here:
#drive.mount('/content/drive')

### Import packages and modules

In [ ]:
from scripts.GCA import train_GCNAE2
from scripts.utils import ball_vol, gen_RGG_edge_index, set_environment_variables
from scripts.test_identification import compute_psi
import torch
import torch_geometric
from torch_geometric.data import Data
import numpy as np
import warnings
from scipy.special import expit

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

### Model Parameters


In [ ]:
# enviroment accessory parameters
parameters['device'] = torch.device("cuda" if torch.cuda.is_available() else "cpu")
parameters['seed'] = 123
parameters['kappa'] = 30
parameters['sample_size'] = 500
parameters['no_simulations'] = 200
parameters['hidden_dim'] = 3
parameters['no_run_network'] = 40
parameters['lr'] = 0.01
parameters['epochs'] = 200
parameters['treatment_prob'] = 0.5 # treatment probability
sign_level = 0.05 # significance level
theta = np.array([-1, 5, 1, 1])

# Learn and test exposure mappings

## Example

* researcher-defined exposure: $\dot{Z}=\frac{\sum_{j \ne i}A_{ij}D_j X_j}{\sum_{j=1}^nA_{ij}} $

* true exposure: $Z=\frac{\sum_{j \ne i}A_{ij}D_j X_j}{\sum_{j=1}^nA_{ij}} $

* $Y_{i} = \alpha  + \delta Z + D_i \gamma + X_i \xi + \varepsilon_i $

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)

parameters = set_environment_variables(parameters)

coef_list = []
pvalues_list = []
cov = []
dot = np.zeros((parameters['no_run_network'],parameters['sample_size']))
tilde = np.zeros((parameters['no_run_network'],parameters['sample_size']))

for i in range(parameters['no_simulations']):

  print('# iteration: ', int(i))

  parameters['seed'] = int(i)
  seed = parameters['seed']
  n = parameters['sample_size']

  positions = np.random.uniform(size=(n,2))
  r = (parameters['kappa']/ball_vol(2,1)/n)**(1/2)
  A_mat, A_norm, edge_index, edge_weight = gen_RGG_edge_index(positions, r)
  errors = np.random.normal(size=n).reshape(-1, 1)
  deg_seq = np.bincount(edge_index[0].numpy(), minlength=n)

  # covariate X
  X = np.random.binomial(1, 0.5, n).reshape(-1, 1)

  # treatment assignment (dependence on X)
  alpha = -1.0
  beta  =  2.0
  p = expit(alpha + beta * X)
  D = np.random.binomial(1, p)

  # researcher-defined exposure -> \dot{Z}
  Z_dot = A_norm.dot(D*X)

  # true exposure Z
  Z = A_norm.dot(D*X)

  # outcome
  Y = theta[0] + theta[1] * Z + theta[2] * D + theta[3] * X + errors

  for k in range(parameters['no_run_network']):

    parameters['seed'] = int(i + k)

    ## Graph Convolutional Autoencoder (learn exposure -> \tilde{Z})
    data = Data(x=np.hstack((D,X)), edge_index=edge_index, edge_weight=edge_weight, y=Y)
    data.x = torch.tensor(data.x, dtype=torch.float32)
    data.x = data.x.to(parameters['device'])
    data.edge_index = data.edge_index.to(parameters['device']).to(torch.int64)
    data.edge_weight = data.edge_weight.to(parameters['device']).to(torch.float32)
    data.y = torch.tensor(data.y, dtype=torch.float32)
    data.y = data.y.to(parameters['device'])
    data.y = data.y.view(-1, 1)
    Z_tilde_iter = train_GCNAE2(parameters, data, parameters["device"])
    tilde[k,:] = Z_tilde_iter.reshape(1,-1)

  Z_tilde = tilde.mean(axis = 0) # \tilde{Z}

  # test the validity of the researcher-defined exposure mapping
  summary = test_trim_reweight(y = Y.flatten(), d = Z_dot.flatten(), z = Z_tilde, MLmethod = 'parametric', k = 4, L = 4, epsilon = 0)
  coef = summary['teststat']
  p_val = summary['pval']

  coef_list.append(coef)
  pvalues_list.append(p_val)
  if pvalues_list[i] < sign_level:
    cov.append(1)
  else:
    cov.append(0)

avg_coef = np.mean(coef_po)
print('mean coef Z:', avg_coef)
print('rejection rate Z:', np.mean(cov))
print('mean p value:', np.mean(pvalues_list))